<a href="https://colab.research.google.com/github/songguan26/InsectRoleVision/blob/main/ScanBugS2_EfficientNetV2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import json

import random
import os
import cv2
from IPython.display import Image
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import classification_report
from collections import Counter

import PIL.Image as Image
import os

import matplotlib.pylab as plt

import tensorflow as tf
import tensorflow_hub as hub

from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential

import keras
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from keras.callbacks import ReduceLROnPlateau
from tensorflow.keras.applications.xception import Xception
from tensorflow.keras.applications.resnet import ResNet50
from tensorflow.keras.applications.resnet import ResNet101
from tensorflow.keras.applications.vgg16 import VGG16
from tensorflow.keras.applications.vgg19 import VGG19
from tensorflow.keras.applications.inception_v3 import InceptionV3
from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2
from tensorflow.keras.applications import EfficientNetV2S
from tensorflow.keras.applications.densenet import DenseNet201
from tensorflow.keras.applications.densenet import DenseNet121
from tensorflow.keras.applications.densenet import DenseNet169
from tensorflow.keras.applications.nasnet import NASNetLarge

from tensorflow.keras.utils import plot_model
from tensorflow.keras.optimizers import Adam, SGD
from keras import Model, layers
from keras.models import Sequential
from keras.layers import GlobalMaxPooling2D, GlobalAveragePooling2D, Dropout, Dense, Input, Conv2D, MaxPooling2D, Flatten,MaxPooling3D

In [ ]:
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


In [ ]:
root_path = '/content/drive/MyDrive/SQ_split_aug_compres'
train_pred_test_folders = os.listdir(root_path)

seg_train_folders = '/content/drive/MyDrive/SQ_split_aug_compres/train'
seg_test_folders = '/content/drive/MyDrive/SQ_split_aug_compres/test'
seg_pred_folders = '/content/drive/MyDrive/SQ_split_aug_compres/pred'

In [ ]:
def do_history_stuff(history, history_file_name, isinception=False):
    save_history(history, history_file_name)
    plot_accuracy_from_history(history, isinception)
    plot_loss_from_history(history)

def save_history(history, model_name):
    #convert the history.history dict to a pandas DataFrame:
    hist_df = pd.DataFrame(history.history)

    # save to json:
    hist_json_file = model_name+'_history.json'
    with open(hist_json_file, mode='w') as f:
        hist_df.to_json(f)

    # or save to csv:
    hist_csv_file = model_name+'_history.csv'
    with open(hist_csv_file, mode='w') as f:
        hist_df.to_csv(f)

def plot_accuracy_from_history(history, isinception=False):
    color = sns.color_palette()
    if(isinception == False):
        acc = history.history['accuracy']
        val_acc = history.history['val_accuracy']
    else:
        acc = history.history['accuracy']
        val_acc = history.history['val_accuracy']


    epochs = range(len(acc))

    sns.lineplot(x=epochs, y=acc, label='Training Accuracy')
    sns.lineplot(x=epochs, y=val_acc,label='Validation Accuracy')
    plt.title('Training and Validation Accuracy')
    plt.legend()
    plt.figure()
    plt.show()

def plot_loss_from_history(history):
    color = sns.color_palette()
    loss = history.history['loss']
    val_loss = history.history['val_loss']

    epochs = range(len(loss))

    sns.lineplot(x=epochs, y=loss,label='Training Loss')
    sns.lineplot(x=epochs, y=val_loss, label='Validation Loss')
    plt.title('Training and Validation Loss')
    plt.legend()
    plt.figure()
    plt.show()

In [ ]:
train_datagen = ImageDataGenerator( rescale = 1.0/255.,shear_range=0.2,zoom_range=0.0)

# we are rescaling by 1.0/255 to normalize the rgb values if they are in range 0-255 the values are too high for good model performance.
train_generator = train_datagen.flow_from_directory(seg_train_folders,
                                                    batch_size=256,
                                                    shuffle=True,
                                                    class_mode='categorical',
                                                    target_size=(224, 224))

validation_datagen = ImageDataGenerator(rescale = 1.0/255.) #we are only normalising to make the prediction, the other parameters were used for agumentation and train weights
validation_generator = validation_datagen.flow_from_directory(seg_test_folders, shuffle=True, batch_size=256, class_mode='categorical', target_size=(224, 224))

test_datagen = ImageDataGenerator(rescale = 1.0/255.)
test_generator = test_datagen.flow_from_directory(seg_pred_folders, shuffle=True, batch_size=256, class_mode='categorical', target_size=(224, 224))

Found 11991 images belonging to 11 classes.
Found 2505 images belonging to 11 classes.
Found 2503 images belonging to 11 classes.


In [ ]:
inv_map_classes = {v: k for k, v in validation_generator.class_indices.items()}
print(validation_generator.class_indices)
print(inv_map_classes)

{'Arachnida-samples': 0, 'Coleoptera-samples': 1, 'Diptera-samples': 2, 'Diptera_Sarcophagidae-samples': 3, 'Diptera_Syrphidae-samples': 4, 'Diptera_Tephritidae_Bactrocera-samples': 5, 'Hemiptera-samples': 6, 'Hymenoptera - Formicidae-samples': 7, 'Hymenoptera-samples': 8, 'Lepidoptera-samples': 9, 'Thysanoptera-samples': 10}
{0: 'Arachnida-samples', 1: 'Coleoptera-samples', 2: 'Diptera-samples', 3: 'Diptera_Sarcophagidae-samples', 4: 'Diptera_Syrphidae-samples', 5: 'Diptera_Tephritidae_Bactrocera-samples', 6: 'Hemiptera-samples', 7: 'Hymenoptera - Formicidae-samples', 8: 'Hymenoptera-samples', 9: 'Lepidoptera-samples', 10: 'Thysanoptera-samples'}


In [ ]:
#strategy one ->adamlr0.001
#freezing convolutional layers; replacing classification layers
EfficientNetV2S_model = EfficientNetV2S(weights='imagenet', include_top=False, input_shape=(224,224,3))
for layers in EfficientNetV2S_model.layers:
            layers.trainable=False

last_output = EfficientNetV2S_model.layers[-1].output
EfficientNetV2S_x = Flatten()(last_output)
EfficientNetV2S_x = Dense(128, activation = 'relu')(EfficientNetV2S_x)
EfficientNetV2S_x = Dropout(0.3)(EfficientNetV2S_x)
EfficientNetV2S_x = Dense(11, activation = 'softmax')(EfficientNetV2S_x)
EfficientNetV2S_final_model = Model(EfficientNetV2S_model.input, EfficientNetV2S_x)
EfficientNetV2S_final_model.compile(optimizer=Adam(learning_rate=0.001), loss='categorical_crossentropy',metrics=['accuracy'])

82420632/82420632 [==============================] - 3s 0us/step


In [ ]:
# EfficientNetV2S
EfficientNetV2S_filepath = 'EfficientNetV2S'+'-saved-model-{epoch:02d}-loss-{loss:.2f}.hdf5'
EfficientNetV2S_checkpoint = tf.keras.callbacks.ModelCheckpoint(EfficientNetV2S_filepath, monitor='val_accuracy', verbose=1, save_best_only=True, mode='max')
EfficientNetV2S_early_stopping = tf.keras.callbacks.EarlyStopping(monitor='loss', patience=5)
EfficientNetV2S_history = EfficientNetV2S_final_model.fit(train_generator, epochs = 100, batch_size=256, validation_data = validation_generator,callbacks=[EfficientNetV2S_checkpoint,EfficientNetV2S_early_stopping],verbose=1)

do_history_stuff(EfficientNetV2S_history, 'EfficientNetB7_model')

Epoch 1/100
47/47 [==============================] - ETA: 0s - loss: 5.5124 - accuracy: 0.1545  
Epoch 1: val_accuracy improved from -inf to 0.19601, saving model to EfficientNetV2S-saved-model-01-loss-5.51.hdf5


/usr/local/lib/python3.10/dist-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


47/47 [==============================] - 10794s 230s/step - loss: 5.5124 - accuracy: 0.1545 - val_loss: 2.3808 - val_accuracy: 0.1960
Epoch 2/100
47/47 [==============================] - ETA: 0s - loss: 2.4349 - accuracy: 0.1872 
Epoch 2: val_accuracy did not improve from 0.19601
47/47 [==============================] - 587s 12s/step - loss: 2.4349 - accuracy: 0.1872 - val_loss: 2.3586 - val_accuracy: 0.1960
Epoch 3/100
47/47 [==============================] - ETA: 0s - loss: 2.3480 - accuracy: 0.1911 
Epoch 3: val_accuracy did not improve from 0.19601
47/47 [==============================] - 590s 13s/step - loss: 2.3480 - accuracy: 0.1911 - val_loss: 2.3370 - val_accuracy: 0.1960
Epoch 4/100
47/47 [==============================] - ETA: 0s - loss: 2.3268 - accuracy: 0.1911 
Epoch 4: val_accuracy did not improve from 0.19601
47/47 [==============================] - 591s 13s/step - loss: 2.3268 - accuracy: 0.1911 - val_loss: 2.3164 - val_accuracy: 0.1960
Epoch 5/100
47/47 [=============

In [ ]:
#pred

EfficientNetV2S_best_model = EfficientNetV2S_final_model

def mode(my_list):
    ct = Counter(my_list)
    max_value = max(ct.values())
    return ([key for key, value in ct.items() if value == max_value])

true_value = []
combined_model_pred = []
EfficientNetV2S_pred = []
for folder in os.listdir(seg_pred_folders):

    test_image_ids = os.listdir(os.path.join(seg_pred_folders,folder))

    for image_id in test_image_ids[:int(len(test_image_ids))]:

        path = os.path.join(seg_pred_folders,folder,image_id)

        true_value.append(validation_generator.class_indices[folder])
        img = cv2.resize(cv2.imread(path),(224,224))
        img_normalized = img/255

        #EfficientNetV2S
        EfficientNetV2S_image_prediction = np.argmax(EfficientNetV2S_best_model.predict(np.array([img_normalized])))
        EfficientNetV2S_pred.append(EfficientNetV2S_image_prediction)

from sklearn.metrics import confusion_matrix
import itertools
#from mlxtend.plotting import plot_confusion_matrix
def clf_report(true_value, model_pred):

    classes = validation_generator.class_indices.keys()
    TP_count = [true_value[i] == model_pred[i] for i in range(len(true_value))]
    model_accuracy = np.sum(TP_count)/len(TP_count)
    print('Model Accuracy', model_accuracy)

    plt.figure(figsize=(7,7))
    cm = confusion_matrix(true_value,model_pred)
    plt.imshow(cm,interpolation='nearest',cmap=plt.cm.viridis)
    plt.title('Confusion Matrix')
    plt.colorbar()
    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes, rotation=45)
    plt.yticks(tick_marks, classes)
    thresh = cm.max()*0.8
    for i,j in itertools.product(range(cm.shape[0]),range(cm.shape[1])):
        plt.text(j,i,cm[i,j],
                horizontalalignment="center",
                color="black" if cm[i,j] > thresh else "white")
        pass

    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    pass

    print(classification_report(true_value, model_pred, target_names = list(classes)))

# EfficientNetV2S model classification report
clf_report(true_value, EfficientNetV2S_pred)

In [ ]:
#strategy one ->adamlr0.0001
#freezing convolutional layers; replacing classification layers
EfficientNetV2S_model = EfficientNetV2S(weights='imagenet', include_top=False, input_shape=(224,224,3))
for layers in EfficientNetV2S_model.layers:
            layers.trainable=False

last_output = EfficientNetV2S_model.layers[-1].output
EfficientNetV2S_x = Flatten()(last_output)
EfficientNetV2S_x = Dense(128, activation = 'relu')(EfficientNetV2S_x)
EfficientNetV2S_x = Dropout(0.3)(EfficientNetV2S_x)
EfficientNetV2S_x = Dense(11, activation = 'softmax')(EfficientNetV2S_x)
EfficientNetV2S_final_model = Model(EfficientNetV2S_model.input, EfficientNetV2S_x)
EfficientNetV2S_final_model.compile(optimizer=Adam(learning_rate=0.0001), loss='categorical_crossentropy',metrics=['accuracy'])

# EfficientNetV2S
EfficientNetV2S_filepath = 'EfficientNetV2S'+'-saved-model-{epoch:02d}-loss-{loss:.2f}.hdf5'
EfficientNetV2S_checkpoint = tf.keras.callbacks.ModelCheckpoint(EfficientNetV2S_filepath, monitor='val_accuracy', verbose=1, save_best_only=True, mode='max')
EfficientNetV2S_early_stopping = tf.keras.callbacks.EarlyStopping(monitor='loss', patience=5)
EfficientNetV2S_history = EfficientNetV2S_final_model.fit(train_generator, epochs = 100, batch_size=256, validation_data = validation_generator,callbacks=[EfficientNetV2S_checkpoint,EfficientNetV2S_early_stopping],verbose=1)

do_history_stuff(EfficientNetV2S_history, 'EfficientNetB7_model')

In [ ]:
#pred

EfficientNetV2S_best_model = EfficientNetV2S_final_model

def mode(my_list):
    ct = Counter(my_list)
    max_value = max(ct.values())
    return ([key for key, value in ct.items() if value == max_value])

true_value = []
combined_model_pred = []
EfficientNetV2S_pred = []
for folder in os.listdir(seg_pred_folders):

    test_image_ids = os.listdir(os.path.join(seg_pred_folders,folder))

    for image_id in test_image_ids[:int(len(test_image_ids))]:

        path = os.path.join(seg_pred_folders,folder,image_id)

        true_value.append(validation_generator.class_indices[folder])
        img = cv2.resize(cv2.imread(path),(224,224))
        img_normalized = img/255

        #EfficientNetV2S
        EfficientNetV2S_image_prediction = np.argmax(EfficientNetV2S_best_model.predict(np.array([img_normalized])))
        EfficientNetV2S_pred.append(EfficientNetV2S_image_prediction)

from sklearn.metrics import confusion_matrix
import itertools
#from mlxtend.plotting import plot_confusion_matrix
def clf_report(true_value, model_pred):

    classes = validation_generator.class_indices.keys()
    TP_count = [true_value[i] == model_pred[i] for i in range(len(true_value))]
    model_accuracy = np.sum(TP_count)/len(TP_count)
    print('Model Accuracy', model_accuracy)

    plt.figure(figsize=(7,7))
    cm = confusion_matrix(true_value,model_pred)
    plt.imshow(cm,interpolation='nearest',cmap=plt.cm.viridis)
    plt.title('Confusion Matrix')
    plt.colorbar()
    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes, rotation=45)
    plt.yticks(tick_marks, classes)
    thresh = cm.max()*0.8
    for i,j in itertools.product(range(cm.shape[0]),range(cm.shape[1])):
        plt.text(j,i,cm[i,j],
                horizontalalignment="center",
                color="black" if cm[i,j] > thresh else "white")
        pass

    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    pass

    print(classification_report(true_value, model_pred, target_names = list(classes)))

# EfficientNetV2S model classification report
clf_report(true_value, EfficientNetV2S_pred)

In [ ]:
#strategy one ->adamlr0.00001
#freezing convolutional layers; replacing classification layers
EfficientNetV2S_model = EfficientNetV2S(weights='imagenet', include_top=False, input_shape=(224,224,3))
for layers in EfficientNetV2S_model.layers:
            layers.trainable=False

last_output = EfficientNetV2S_model.layers[-1].output
EfficientNetV2S_x = Flatten()(last_output)
EfficientNetV2S_x = Dense(128, activation = 'relu')(EfficientNetV2S_x)
EfficientNetV2S_x = Dropout(0.3)(EfficientNetV2S_x)
EfficientNetV2S_x = Dense(11, activation = 'softmax')(EfficientNetV2S_x)
EfficientNetV2S_final_model = Model(EfficientNetV2S_model.input, EfficientNetV2S_x)
EfficientNetV2S_final_model.compile(optimizer=Adam(learning_rate=0.00001), loss='categorical_crossentropy',metrics=['accuracy'])

# EfficientNetV2S
EfficientNetV2S_filepath = 'EfficientNetV2S'+'-saved-model-{epoch:02d}-loss-{loss:.2f}.hdf5'
EfficientNetV2S_checkpoint = tf.keras.callbacks.ModelCheckpoint(EfficientNetV2S_filepath, monitor='val_accuracy', verbose=1, save_best_only=True, mode='max')
EfficientNetV2S_early_stopping = tf.keras.callbacks.EarlyStopping(monitor='loss', patience=5)
EfficientNetV2S_history = EfficientNetV2S_final_model.fit(train_generator, epochs = 100, batch_size=256, validation_data = validation_generator,callbacks=[EfficientNetV2S_checkpoint,EfficientNetV2S_early_stopping],verbose=1)

do_history_stuff(EfficientNetV2S_history, 'EfficientNetB7_model')

In [ ]:
#pred

EfficientNetV2S_best_model = EfficientNetV2S_final_model

def mode(my_list):
    ct = Counter(my_list)
    max_value = max(ct.values())
    return ([key for key, value in ct.items() if value == max_value])

true_value = []
combined_model_pred = []
EfficientNetV2S_pred = []
for folder in os.listdir(seg_pred_folders):

    test_image_ids = os.listdir(os.path.join(seg_pred_folders,folder))

    for image_id in test_image_ids[:int(len(test_image_ids))]:

        path = os.path.join(seg_pred_folders,folder,image_id)

        true_value.append(validation_generator.class_indices[folder])
        img = cv2.resize(cv2.imread(path),(224,224))
        img_normalized = img/255

        #EfficientNetV2S
        EfficientNetV2S_image_prediction = np.argmax(EfficientNetV2S_best_model.predict(np.array([img_normalized])))
        EfficientNetV2S_pred.append(EfficientNetV2S_image_prediction)

from sklearn.metrics import confusion_matrix
import itertools
#from mlxtend.plotting import plot_confusion_matrix
def clf_report(true_value, model_pred):

    classes = validation_generator.class_indices.keys()
    TP_count = [true_value[i] == model_pred[i] for i in range(len(true_value))]
    model_accuracy = np.sum(TP_count)/len(TP_count)
    print('Model Accuracy', model_accuracy)

    plt.figure(figsize=(7,7))
    cm = confusion_matrix(true_value,model_pred)
    plt.imshow(cm,interpolation='nearest',cmap=plt.cm.viridis)
    plt.title('Confusion Matrix')
    plt.colorbar()
    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes, rotation=45)
    plt.yticks(tick_marks, classes)
    thresh = cm.max()*0.8
    for i,j in itertools.product(range(cm.shape[0]),range(cm.shape[1])):
        plt.text(j,i,cm[i,j],
                horizontalalignment="center",
                color="black" if cm[i,j] > thresh else "white")
        pass

    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    pass

    print(classification_report(true_value, model_pred, target_names = list(classes)))

# EfficientNetV2S model classification report
clf_report(true_value, EfficientNetV2S_pred)